In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models

In [ ]:
DWT_TRAIN_DIR = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/tow_ids/Preprocessing/hasil/imgsize228/norm/dwt/"
DWT_TEST_DIR  = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/tow_ids/Preprocessing/hasil/imgsize228/norm/dwt/"

In [ ]:
# Cell 3 — Factorized convolution block

def factorized_conv(x, filters1, filters2):
    
    # 1x3 convolution
    x = composite_conv(
        x,
        filters=filters1,
        kernel_size=(1,3)
    )
    
    # 3x1 convolution
    x = composite_conv(
        x,
        filters=filters2,
        kernel_size=(3,1)
    )
    
    return x

In [ ]:
# Cell 4 — DGC block

def DGC_block(x):

    # =========================
    # Branch 1
    # =========================
    
    branch1_a = composite_conv(
        x,
        filters=16,
        kernel_size=(1,3)
    )
    
    branch1_b = composite_conv(
        x,
        filters=16,
        kernel_size=(3,1)
    )
    
    concat = layers.Concatenate()([branch1_a, branch1_b])
    
    
    conv1 = composite_conv(
        concat,
        filters=32,
        kernel_size=(1,1)
    )
    
    
    pool1 = layers.MaxPooling2D(
        pool_size=(2,2)
    )(conv1)
    
    
    group_conv1 = composite_conv(
        pool1,
        filters=64,
        kernel_size=(3,1),
        groups=4
    )
    

    # =========================
    # Residual branch
    # =========================
    
    res = composite_conv(
        x,
        filters=32,
        kernel_size=(1,1)
    )
    
    res = layers.MaxPooling2D(
        pool_size=(2,2)
    )(res)
    
    
    group_conv2 = composite_conv(
        res,
        filters=64,
        kernel_size=(1,3),
        groups=4
    )
    
    
    # =========================
    # Merge
    # =========================
    
    out = layers.Add()([
        group_conv1,
        group_conv2
    ])
    
    
    return out

In [ ]:
# Cell 5 — Input layer

input_layer = layers.Input(shape=(228,228,3))

In [ ]:
# Cell 6 — First feature extractor

x = factorized_conv(
    input_layer,
    filters1=32,
    filters2=16
)

In [ ]:
# Cell 7 — MaxPooling

x = layers.MaxPooling2D(
    pool_size=(2,2)
)(x)

In [ ]:
# Cell 8 — DGC block 1

x = DGC_block(x)

In [ ]:
# Cell 9 — DGC block 2

x = DGC_block(x)

In [ ]:
# Cell 10 — 1x1 convolution

x = composite_conv(
    x,
    filters=32,
    kernel_size=(1,1)
)

In [ ]:
# Cell 11 — Average Pooling

x = layers.GlobalAveragePooling2D()(x)

In [ ]:
# Cell 12 — Flatten

x = layers.Flatten()(x)

In [ ]:
# Cell 13 — Fully connected layer

x = layers.Dense(
    128,
    activation='relu'
)(x)

In [ ]:
# Cell 14 — Output layer

output_layer = layers.Dense(
    6,
    activation='softmax'
)(x)

In [ ]:
# Cell 15 — Build model

model = models.Model(
    inputs=input_layer,
    outputs=output_layer
)

model.summary()